In [6]:
import pandas as pd

df = pd.read_csv('m2_metric_scenarios.csv')

tp = df['true_positive']
fp = df['false_positive']
fn = df['false_negative']
tn = df['true_negative']

df['my_accuracy'] = (tp + tn) / (tp + fp + fn + tn)
df['my_precision'] = tp / (tp + fp)
df['my_recall'] = tp / (tp + fn)
df['my_f1'] = (2 * tp) / (2 * tp + fp + fn)

# compare my calculated metrics with the csv file's precomputed columns
display(df[[
    'scenario_id',
    'accuracy', 'my_accuracy',
    'precision', 'my_precision',
    'recall', 'my_recall',
    'f1', 'my_f1'
]].round(4))

,scenario_id,accuracy,my_accuracy,precision,my_precision,recall,my_recall,f1,my_f1
0,S1,0.9900,0.9900,0.8333,0.8333,0.9524,0.9524,0.8889,0.8889
1,S2,0.9918,0.9918,0.9266,0.9266,0.7947,0.7947,0.8556,0.8556
2,S3,0.9930,0.9930,0.8757,0.8757,0.9226,0.9226,0.8986,0.8986
3,S4,0.9536,0.9536,0.8000,0.8000,0.8455,0.8455,0.8221,0.8221


In [7]:
df['total_error_cost'] = (
    df['false_positive'] * df['dollar_cost_per_false_positive']
    + df['false_negative'] * df['dollar_cost_per_false_negative']
)

# show highest to lowest total error cost
print("Scenarios ranked by total error cost (highest first):")
print(df[['scenario_id', 'accuracy', 'total_error_cost']]
      .sort_values('total_error_cost', ascending=False))

print()  # Blank line between tables.

# show highest to lowest accuracy, to make a comparison
print("Scenarios ranked by accuracy (highest first):")
print(df[['scenario_id', 'accuracy', 'total_error_cost']]
      .sort_values('accuracy', ascending=False))

Scenarios ranked by total error cost (highest first):
  scenario_id  accuracy  total_error_cost
0          S1    0.9900            500008
3          S4    0.9536            343000
2          S3    0.9930             27140
1          S2    0.9918              7520

Scenarios ranked by accuracy (highest first):
  scenario_id  accuracy  total_error_cost
2          S3    0.9930             27140
1          S2    0.9918              7520
0          S1    0.9900            500008
3          S4    0.9536            343000


In [8]:
fp, fn = 130, 95
cost_fp, cost_fn = 300, 3200

print("S4: Threshold settings ranked by sweep step")
print(f"{'Step':>4} {'False positives':>16} {'False negatives':>16} {'Total error cost':>18}")

best_step = None
lowest_cost = float('inf')

for step in range(0, 6):
    f_p = fp + 4 * step
    f_n = fn - step
    total = f_p * cost_fp + f_n * cost_fn

    print(f"{step:>4} {f_p:>16} {f_n:>16} ${total:>17,.0f}")

    if total < lowest_cost:
        lowest_cost = total
        best_step = step

print()
print(f"Lowest cost among tested settings: step {best_step}, "
      f"with total error cost of ${lowest_cost:,.0f}.")

S4: Threshold settings ranked by sweep step
Step  False positives  False negatives   Total error cost
   0              130               95 $          343,000
   1              134               94 $          341,000
   2              138               93 $          339,000
   3              142               92 $          337,000
   4              146               91 $          335,000
   5              150               90 $          333,000

Lowest cost among tested settings: step 5, with total error cost of $333,000.
